# **NOTEBOOK 2: Preprocessing**

In [2]:
import pandas as pd
from pathlib import Path

## Load Dataset

In [544]:
def load_dataset():
    file_path = Path.home() / 'Documents/IDX Internship/Raw Data'
    
    # all 30 datasets
    files = [
        file_path / 'CRMLSSold20220101_20231231_filled.csv', file_path / 'CRMLSSold202401_filled.csv',
        file_path / 'CRMLSSold202402_filled.csv', file_path / 'CRMLSSold202403_filled.csv',
        file_path / 'CRMLSSold202404_filled.csv', file_path / 'CRMLSSold202405_filled.csv',
        file_path / 'CRMLSSold202406_filled.csv', file_path / 'CRMLSSold202407_filled.csv',
        file_path / 'CRMLSSold202408.csv', file_path / 'CRMLSSold202409.csv',
        file_path / 'CRMLSSold202410.csv', file_path / 'CRMLSSold202411.csv',
        file_path / 'CRMLSSold202412.csv', file_path / 'CRMLSSold202501_filled.csv',
        file_path / 'CRMLSSold202502.csv', file_path / 'CRMLSSold202503.csv',
        file_path / 'CRMLSSold202504.csv', file_path / 'CRMLSSold202505.csv',
        file_path / 'CRMLSSold202506.csv', file_path / 'CRMLSSold202507.csv',
        file_path / 'CRMLSSold202508.csv', file_path / 'CRMLSSold202509.csv',
        file_path / 'CRMLSSold202510.csv', file_path / 'CRMLSSold202511.csv',
        file_path / 'CRMLSSold202512.csv', file_path / 'CRMLSSold202601.csv',
        file_path / 'CRMLSSold202602.csv', file_path / 'CRMLSSold202603.csv',
        file_path / 'CRMLSSold202604.csv', file_path / 'CRMLSSold202605.csv',
    ]
    
    # dtype to str to handle mixed value types for some columns
    data_dfs = [pd.read_csv(f, dtype={'WaterfrontYN': str, 'PostalCode': str, 'latfilled': str, 'lonfilled': str,
                                      'ElementarySchool': str, 'BuilderName': str, 'CoBuyerAgentFirstName': str,
                                      'PoolPrivateYN': str}) for f in files]
    df = pd.concat(data_dfs, ignore_index=True)

    return df

df = load_dataset()
print(df.shape)

(794271, 82)


## Filter Dataset
* PropertyType = Residential
* PropertySubType = SingleFamily Residence

In [26]:
# restrict analysis to residential and single family
df = df[
    (df['PropertyType'] == 'Residential') &
    (df['PropertySubType'] == 'SingleFamilyResidence')].copy()

print(df.shape)

(399157, 82)


## Convert Dates

In [511]:
df['CloseDate'] = pd.to_datetime(df['CloseDate'])
df['CloseDate']

0        2022-02-25
1        2022-02-19
2        2022-04-15
3        2022-01-04
4        2022-01-12
            ...    
794266   2026-05-28
794267   2026-05-21
794268   2026-05-11
794269   2026-05-11
794270   2026-05-01
Name: CloseDate, Length: 794271, dtype: datetime64[ns]

## Handle Missing Values
* Drop columns not known at prediction time
* Drop columns with null rate above 50%
* Drop columns with no plausible relationship to ClosePrice
* Fill columns with less than 50% null rate with median or sentinel
* Consider dropping columns with correlation to ClosePrice > -0.05

In [512]:
# count missing values
missing = pd.DataFrame({
    'Total Missing Values': df.isnull().sum(),
    'Proportion Missing': df.isnull().mean()
})

missing.sort_values("Proportion Missing", ascending=False)

,Total Missing Values,Proportion Missing
CoveredSpaces,794271,1.000000
AboveGradeFinishedArea,794271,1.000000
FireplacesTotal,794271,1.000000
MiddleOrJuniorSchoolDistrict,794271,1.000000
ElementarySchoolDistrict,794271,1.000000
...,...,...
ListingId,1,0.000001
CountyOrParish,1,0.000001
CloseDate,0,0.000000
ListingKey,0,0.000000


In [549]:
preprocessed_df = df.copy()
# preprocessed_df.columns

In [514]:
# drop columns not known at prediction time
drop_cols = ['ListPrice', 'DaysOnMarket', 'OriginalListPrice', 'CloseDate']
preprocessed_df = preprocessed_df.drop(columns=drop_cols)
preprocessed_df.shape

(794271, 78)

In [515]:
# drop column if null rate above 50%
threshold = 0.5

drop_cols = missing[missing['Proportion Missing'] > threshold].index.tolist()

preprocessed_df = preprocessed_df.drop(columns=drop_cols)
preprocessed_df.shape

(794271, 51)

In [516]:
# preprocessed_df.columns

In [517]:
# drop columns with no plausible relationship to ClosePrice
drop_cols = ['ContractStatusChangeDate', 'MlsStatus', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'ListingId',
             'PurchaseContractDate', 'ListingContractDate', 'ListAgentEmail', 'ListOfficeName', 'BuyerOfficeName',
             'ListingKey', 'ListingKeyNumeric', 'ListAgentFirstName', 'ListAgentLastName', 'StreetNumberNumeric',
             'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerOfficeAOR', 'BuyerAgentAOR', 'ListAgentAOR', 'UnparsedAddress']
preprocessed_df = preprocessed_df.drop(columns=drop_cols)
preprocessed_df.shape

(794271, 30)

In [518]:
# fill values with less than 50% null rate

# sentinel fill
preprocessed_df['ParkingTotal'] = preprocessed_df['ParkingTotal'].fillna(0)
preprocessed_df['GarageSpaces'] = preprocessed_df['GarageSpaces'].fillna(0)
preprocessed_df['MainLevelBedrooms'] = preprocessed_df['MainLevelBedrooms'].fillna(0)

# median fill
for col in ['LotSizeSquareFeet', 'LotSizeArea', 'YearBuilt', 'Stories', 'Latitude', 'Longitude', 'BedroomsTotal',
            'BathroomsTotalInteger']:
    preprocessed_df[col] = preprocessed_df[col].fillna(preprocessed_df[col].median())

In [519]:
missing_rate = preprocessed_df.isnull().mean()

cols = missing_rate[(missing_rate > 0) & (missing_rate <= 0.50)].sort_values(ascending=False)

print(cols)

Flooring              0.413442
HighSchoolDistrict    0.333918
AssociationFee        0.313009
AttachedGarageYN      0.277589
Levels                0.151297
NewConstructionYN     0.140315
PoolPrivateYN         0.131257
MLSAreaMajor          0.111283
ViewYN                0.109527
LotSizeAcres          0.095071
FireplaceYN           0.094303
PropertySubType       0.078646
LivingArea            0.072536
City                  0.001228
PostalCode            0.000291
ClosePrice            0.000010
StateOrProvince       0.000003
CountyOrParish        0.000001
dtype: float64


In [520]:
# create binary has_hoa feature from AssociationFee
preprocessed_df['AssociationFee'] = preprocessed_df['AssociationFee'].fillna(0)
preprocessed_df["has_hoa"] = (preprocessed_df["AssociationFee"] > 0).astype(int)

In [521]:
preprocessed_df[['has_hoa', 'AssociationFee']].head()

,has_hoa,AssociationFee
0,0,0.0
1,0,0.0
2,1,370.0
3,1,140.0
4,1,300.0


In [522]:
preprocessed_df.columns

Index(['Flooring', 'ViewYN', 'PoolPrivateYN', 'ClosePrice', 'Latitude',
       'Longitude', 'PropertyType', 'LivingArea', 'MLSAreaMajor',
       'CountyOrParish', 'AttachedGarageYN', 'ParkingTotal', 'PropertySubType',
       'LotSizeAcres', 'YearBuilt', 'BathroomsTotalInteger', 'City',
       'BedroomsTotal', 'StateOrProvince', 'FireplaceYN', 'Stories', 'Levels',
       'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces',
       'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'has_hoa'],
      dtype='object')

In [523]:
# drop features with zero variance
preprocessed_df = preprocessed_df.drop(columns=['PropertyType', 'PropertySubType'])

In [524]:
preprocessed_df['StateOrProvince'].unique()

array(['CA', 'BC', 'UT', nan, 'AL', 'WA', 'AZ', 'OK', 'VA', 'OS', 'NV',
       'HI', 'TX', 'LA', 'OH', 'ID', 'GA', 'ME', 'NY', 'CO', 'MO', 'MI',
       'TN', 'AR', 'MT', 'FL', 'NJ', 'OR'], dtype=object)

In [525]:
preprocessed_df[['LotSizeSquareFeet', 'LotSizeArea', 'LotSizeAcres']].count()

LotSizeSquareFeet    794271
LotSizeArea          794271
LotSizeAcres         718759
dtype: int64

In [526]:
# keep only one area column (LotSizeSquareFeet)
preprocessed_df = preprocessed_df.drop(columns=['LotSizeArea', 'LotSizeAcres'])
preprocessed_df.shape

(794271, 27)

In [527]:
# fill null values for certain bool values with False
for col in ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']:
    preprocessed_df[col] = preprocessed_df[col].astype(bool)
    preprocessed_df[col] = preprocessed_df[col].fillna(False)

preprocessed_df['ViewYN'].dtype

dtype('bool')

In [528]:
# fill missing values for certain categorical variables with 'Unknown'
for col in ['Flooring', 'HighSchoolDistrict', 'MLSAreaMajor', 'Levels', 'City']:
    preprocessed_df[col] = preprocessed_df[col].fillna('Unknown')

In [529]:
preprocessed_df['StateOrProvince'].value_counts()

StateOrProvince
CA    794177
AZ        19
NV         9
OK         8
OS         8
TX         6
FL         5
ID         4
AL         4
HI         4
MO         3
CO         3
AR         2
NY         2
OH         2
VA         2
GA         1
ME         1
BC         1
LA         1
MI         1
TN         1
WA         1
MT         1
UT         1
NJ         1
OR         1
Name: count, dtype: int64

In [550]:
# replace StateOrProvince (almost no variance) with binary InCalifornia variable
preprocessed_df["InCalifornia"] = (preprocessed_df["StateOrProvince"] == "CA").astype(int)
preprocessed_df = preprocessed_df.drop(columns=['StateOrProvince'])

# fill missing value for CountyOrParish with mode
preprocessed_df['CountyOrParish'] = preprocessed_df['CountyOrParish'].fillna(preprocessed_df['CountyOrParish'].mode()[0])

In [531]:
preprocessed_df = preprocessed_df.dropna(subset=['ClosePrice', 'LivingArea', 'PostalCode'])

## Data Preprocessing Function

In [602]:
def preprocessing():

    # load dataset
    df = load_dataset()
    
    # restrict analysis to residential and single family
    df = df[
        (df['PropertyType'] == 'Residential') &
        (df['PropertySubType'] == 'SingleFamilyResidence')].copy()

    # convert CloseDate to datetime dtype
    df['CloseDate'] = pd.to_datetime(df['CloseDate'])


    preprocessed_df = df.copy()
    
    
    # drop columns not known at prediction time
    drop_cols = ['ListPrice', 'DaysOnMarket', 'OriginalListPrice']
    preprocessed_df = preprocessed_df.drop(columns=drop_cols)

    # drop column if null rate above 50%
    threshold = 0.5
    null_50 = missing[missing['Proportion Missing'] > threshold].index.tolist()
    preprocessed_df = preprocessed_df.drop(columns=null_50)

    # drop columns with no plausible relationship to ClosePrice
    unrelated = ['ContractStatusChangeDate', 'MlsStatus', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'ListingId',
                 'PurchaseContractDate', 'ListingContractDate', 'ListAgentEmail', 'ListOfficeName', 'BuyerOfficeName',
                 'ListingKey', 'ListingKeyNumeric', 'ListAgentFirstName', 'ListAgentLastName', 'StreetNumberNumeric',
                 'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerOfficeAOR', 'BuyerAgentAOR', 'ListAgentAOR',
                 'UnparsedAddress']
    preprocessed_df = preprocessed_df.drop(columns=unrelated)

    # fill values with less than 50% null rate
    # sentinel fill
    preprocessed_df['ParkingTotal'] = preprocessed_df['ParkingTotal'].fillna(0)
    preprocessed_df['GarageSpaces'] = preprocessed_df['GarageSpaces'].fillna(0)
    preprocessed_df['MainLevelBedrooms'] = preprocessed_df['MainLevelBedrooms'].fillna(0)
    
    # median fill
    for col in ['LotSizeSquareFeet', 'YearBuilt', 'Stories', 'Latitude', 'Longitude', 'BedroomsTotal',
                'BathroomsTotalInteger']:
        preprocessed_df[col] = preprocessed_df[col].fillna(preprocessed_df[col].median())

    # create binary has_hoa feature from AssociationFee
    preprocessed_df['AssociationFee'] = preprocessed_df['AssociationFee'].fillna(0)
    preprocessed_df["has_hoa"] = (preprocessed_df["AssociationFee"] > 0).astype(int)
    
    # drop features with zero variance
    preprocessed_df = preprocessed_df.drop(columns=['PropertyType', 'PropertySubType'])

    # keep only one area column (LotSizeSquareFeet)
    preprocessed_df = preprocessed_df.drop(columns=['LotSizeArea', 'LotSizeAcres'])

    # fill null values for certain bool values with False
    for col in ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']:
        preprocessed_df[col] = preprocessed_df[col].astype(bool)
        preprocessed_df[col] = preprocessed_df[col].fillna(False).astype(int)

    # fill missing values for certain categorical variables with 'Unknown'
    for col in ['Flooring', 'HighSchoolDistrict', 'MLSAreaMajor', 'Levels', 'City']:
        preprocessed_df[col] = preprocessed_df[col].fillna('Unknown')

    # replace StateOrProvince (almost no variance) with binary InCalifornia variable
    preprocessed_df["InCalifornia"] = (preprocessed_df["StateOrProvince"] == "CA").astype(int)
    preprocessed_df = preprocessed_df.drop(columns=['StateOrProvince'])
    
    # fill missing value for CountyOrParish with mode
    preprocessed_df['CountyOrParish'] = preprocessed_df['CountyOrParish'].fillna(
        preprocessed_df['CountyOrParish'].mode()[0]
    )

    # drop rows with missing values for target variable and other important features with very little null values
    preprocessed_df = preprocessed_df.dropna(subset=['ClosePrice', 'LivingArea', 'PostalCode'])

    # multi-label encode Flooring feature
    preprocessed_df['Flooring_SeeRemarks'] = (preprocessed_df['Flooring'].str.contains('SeeRemarks').astype(int))
    preprocessed_df['Flooring'] = (preprocessed_df['Flooring'].str.replace('SeeRemarks,?', '', regex=True))
    flooring_dummies = (preprocessed_df['Flooring'].str.get_dummies(sep=','))
    
    preprocessed_df = pd.concat(
        [preprocessed_df.drop(columns=['Flooring']), flooring_dummies.add_prefix('Flooring_')], axis=1)

    # frequency encode MLSAreaMajor
    area_frequency = preprocessed_df['MLSAreaMajor'].value_counts()
    preprocessed_df['MLSAreaMajor_freq'] = (preprocessed_df['MLSAreaMajor'].map(area_frequency))
    
    preprocessed_df = preprocessed_df.drop(columns=['MLSAreaMajor'])

    # one-hot encode CountyOrParish
    preprocessed_df['CountyOrParish'] = (preprocessed_df['CountyOrParish'].str.title()) # standard capitalization
    preprocessed_df['CountyOrParish'] = (preprocessed_df['CountyOrParish'].replace(
        ['Foreign Country', 'Other County', 'Other State', 'Other'], 'Other')) # replace non-counties with 'Other'
    preprocessed_df = pd.get_dummies(preprocessed_df,columns=['CountyOrParish'],drop_first=True)

    # multi-label encode Levels
    # create multi-label dummy variables
    levels_dummies = (preprocessed_df['Levels'].str.get_dummies(sep=','))
    levels_dummies = levels_dummies.add_prefix('Levels_') # add prefix
    
    preprocessed_df = pd.concat([preprocessed_df.drop(columns=['Levels']),levels_dummies],axis=1)

    # group cities with less than 500 entires and one-hot encode City
    city_counts = preprocessed_df['City'].value_counts()
    rare_cities = city_counts[city_counts < 500].index
    preprocessed_df['City'] = (preprocessed_df['City'].replace(rare_cities, 'Other'))
    preprocessed_df = pd.get_dummies(preprocessed_df, columns=['City'], drop_first=True)

    # group districts with less than 100 entries and one-hot encode HighSchoolDistrict
    district_counts = preprocessed_df['HighSchoolDistrict'].value_counts()
    rare_districts = district_counts[district_counts < 100].index
    preprocessed_df['HighSchoolDistrict'] = (preprocessed_df['HighSchoolDistrict'].replace(rare_districts, 'Other'))
    preprocessed_df = pd.get_dummies(preprocessed_df, columns=['HighSchoolDistrict'], drop_first=True)

    # frequency encode PostalCode
    postal_counts = preprocessed_df['PostalCode'].value_counts()
    preprocessed_df['PostalCode_freq'] = (preprocessed_df['PostalCode'].map(postal_counts))
    preprocessed_df = preprocessed_df.drop(columns=['PostalCode'])

    return preprocessed_df

preprocessed_df = preprocessing()
preprocessed_df.shape

(398943, 563)

In [593]:
district_counts = preprocessed_df['HighSchoolDistrict'].value_counts()

rare_districts = district_counts[district_counts < 100].index

preprocessed_df['HighSchoolDistrict'] = (preprocessed_df['HighSchoolDistrict'].replace(rare_districts, 'Other'))
preprocessed_df['HighSchoolDistrict'].value_counts()

HighSchoolDistrict
Unknown                            103116
Other                               37298
Los Angeles Unified                 23310
Capistrano Unified                   6863
Riverside Unified                    5166
                                    ...  
Needles Unified School District       114
East Whittier Unified                 114
Vallejo City Unified                  114
Borrego Springs Unified               111
La Jolla                              101
Name: count, Length: 237, dtype: int64

In [599]:
preprocessed_df['PostalCode'].unique().size

3673

In [601]:
preprocessed_df.columns

Index(['ViewYN', 'PoolPrivateYN', 'ClosePrice', 'Latitude', 'Longitude',
       'LivingArea', 'AttachedGarageYN', 'ParkingTotal', 'YearBuilt',
       'BathroomsTotalInteger',
       ...
       'HighSchoolDistrict_Westminster Unified',
       'HighSchoolDistrict_Westside Union',
       'HighSchoolDistrict_Whittier Union High',
       'HighSchoolDistrict_William S. Hart Union',
       'HighSchoolDistrict_Willows Unified',
       'HighSchoolDistrict_Wiseburn Unified',
       'HighSchoolDistrict_Yosemite Unified',
       'HighSchoolDistrict_Yucaipa/Calimesa Unified',
       'HighSchoolDistrict_Yucca Valley', 'PostalCode_freq'],
      dtype='object', length=562)

## Train-Test Split
* Most recent month as test dataset
* X months prior used as test dataset
* Experiment with different values of X

In [604]:
def train_test_split(df, train_months):
    """
    Split MLS data into training and testing sets.

    Parameters:
        df: dataframe containing CloseDate
        train_months: number of months before test month to use for training

    Returns:
        X_train, X_test, y_train, y_test
    """

    df = df.copy()

    # Ensure datetime
    df['CloseDate'] = pd.to_datetime(df['CloseDate'])

    # Get monthly periods
    df['CloseMonth'] = df['CloseDate'].dt.to_period('M')

    # Most recent month becomes test
    latest_month = df['CloseMonth'].max()

    test_df = df[
        df['CloseMonth'] == latest_month
    ]

    # X months immediately before test month
    train_start = latest_month - train_months

    train_df = df[
        (df['CloseMonth'] < latest_month) &
        (df['CloseMonth'] >= train_start)
    ]

    # Remove date columns
    train_df = train_df.drop(
        columns=['CloseDate', 'CloseMonth']
    )

    test_df = test_df.drop(
        columns=['CloseDate', 'CloseMonth']
    )

    # Separate target
    X_train = train_df.drop(columns=['ClosePrice'])
    y_train = train_df['ClosePrice']

    X_test = test_df.drop(columns=['ClosePrice'])
    y_test = test_df['ClosePrice']

    return X_train, X_test, y_train, y_test

In [605]:
for months in [3, 6, 9, 12, 18, 24]:
    X_train, X_test, y_train, y_test = time_split(preprocessed_df, train_months=months)

    print(f'{months} months:', 'Train:', X_train.shape, 'Test:', X_test.shape)

3 months: Train: (31749, 561) Test: (12016, 561)
6 months: Train: (59415, 561) Test: (12016, 561)
9 months: Train: (94332, 561) Test: (12016, 561)
12 months: Train: (129901, 561) Test: (12016, 561)
18 months: Train: (190669, 561) Test: (12016, 561)
24 months: Train: (266074, 561) Test: (12016, 561)
